In [1]:
import numpy as np
from sympy.physics.quantum import Dagger
from sympy.core.singleton import S
import sympy
import scipy
from sympy.physics.quantum.boson import BosonOp
from sympy.physics.quantum.fermion import FermionOp
from second_quantization import fermions, bosons

In [2]:
def basis_operators(operators, sparse, truncation):
    fermion_ops, boson_ops = [], []
    for operator in operators:
        if isinstance(operator, sympy.physics.quantum.fermion.FermionOp):
            fermion_ops.append(operator)
        elif isinstance(operator, sympy.physics.quantum.boson.BosonOp):
            boson_ops.append(operator)
        else:
            raise ValueError("Can't submit non operator types")
    if not truncation and len(boson_ops) > 0:
        raise ValueError("Please provide a truncation for your bosons")

    fermion_dict = (
        fermions.fermion_basis(fermion_ops, sparse)
        if len(fermion_ops) != 0
        else {1: np.array([1])}
    )
    boson_dict = (
        bosons.boson_basis(boson_ops, truncation, sparse)
        if len(boson_ops) != 0
        else {1: np.array([1])}
    )
    return fermion_dict, boson_dict


def to_matrix(
    expression: sympy.Expr,
    operators: list[
        sympy.physics.quantum.fermion.FermionOp | sympy.physics.quantum.boson.BosonOp
    ],
    sparse: bool,
    operator_dict: dict = None,
    truncation: int | list[int] = None,
) -> dict[sympy.Expr, np.ndarray | scipy.sparse.csr_array]:
    """Transform a sympy expression containing fermionic operators to matrix representation.

    Takes a symbolic expression with fermionic operators and converts it to a dictionary
    containing the matrix representation of each term, grouped by symbolic coefficients.

    Args:
        expression: A sympy expression containing fermionic operators.
        operators: A list of fermionic operators defining the basis.
        sparse: If True, the matrices will be in sparse format. Otherwise, they will be dense.

    Returns:
        A dictionary where the keys are sympy expressions (symbolic coefficients) and
        the values are the corresponding matrix representations in either dense or sparse format.

    Example:
        ```python
        import sympy as sy
        from sympy.physics.quantum.fermion import FermionOp
        from sympy.physics.quantum import Dagger

        # Define symbolic parameters and operators
        t = sy.Symbol('t')
        c = FermionOp('c')
        d = FermionOp('d')

        # Define an expression
        expr = t * Dagger(c) * d

        # Convert to matrix form
        matrices = to_matrix(expr, [c, d], sparse=False)
        ```
    """

    if operator_dict is None:
        fermion_dict, boson_dict = basis_operators(operators, sparse, truncation)

    kron = scipy.sparse.kron if sparse else np.kron

    dict_matrices = {}
    # Expand the expression to ensure all terms are separated
    expression = expression.expand()
    # Iterate over each term in the expression
    for term, coeff in expression.as_coefficients_dict().items():
        term_mat = float(coeff)
        symbol = S.One

        fermion_factors = fermion_dict[1].copy()
        boson_factors = boson_dict[1].copy()

        for factor in term.as_ordered_factors():
            if factor in fermion_dict:
                fermion_factors = fermion_factors @ fermion_dict[factor]
            elif Dagger(factor) in fermion_dict:
                fermion_factors = (
                    fermion_factors @ fermion_dict[Dagger(factor)].conj().T
                )
            elif factor in boson_dict:
                boson_factors = boson_factors @ boson_dict[factor]
            elif Dagger(factor) in boson_dict:
                boson_factors = boson_factors @ boson_dict[Dagger(factor)].conj().T
            else:
                symbol *= factor

        term_mat = term_mat * kron(fermion_factors, boson_factors)

        if symbol in dict_matrices.keys():
            dict_matrices[symbol] += term_mat
        else:
            dict_matrices[symbol] = term_mat
    return dict_matrices

In [3]:
b_0, b_1 = [BosonOp("b_{}".format(i)) for i in range(2)]
f_0, f_1 = [FermionOp("f_{}".format(i)) for i in range(2)]

In [4]:
g1, g2, g3 = sympy.symbols("g1 g2 g3", real=True, commutative=True)

In [10]:
H_int = g1 * Dagger(b_0) * f_0 + g2 * Dagger(f_0) * b_0 + g3
H_int

g1*Dagger(b_0)*f_0 + g2*Dagger(f_0)*b_0 + g3